# BasketballGAN — Training on Google Colab (PyTorch)

Train a Diffusion model (DDPM+DDIM) to generate defensive basketball play simulations.

> **Before starting:** Runtime → Change runtime type → **T4 GPU**

## 1. Clone & Install

In [ ]:
!rm -rf basketball
!git clone https://github.com/TQG1997/basketball.git
%cd basketball
!pip install -q torch numpy matplotlib gradio gdown

## 2. Download Dataset

Downloads the required `.npy` files (~730MB) from Google Drive via `gdown`.

In [ ]:
import os
os.makedirs('data', exist_ok=True)

!pip install -q gdown

FOLDER_ID = '1uNPw7LOA3xENclQRtSlUftiR7tlVNOts'
!gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O data/

# Move files up from subdirectory
import glob, shutil
for f in glob.glob('data/*/*.npy'):
    shutil.move(f, 'data/')
!rmdir data/*/ 2>/dev/null

print('\nDataset files:')
!ls -lh data/

## 3. Mount Google Drive & Check for Previous Training

Detects if you have existing checkpoints from a previous session and lets you resume or start fresh.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, re

DRIVE_OUTPUT = '/content/drive/MyDrive/basketballgan/output'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Check for existing checkpoints
ckpt_dir = os.path.join(DRIVE_OUTPUT, 'Checkpoints')
existing = sorted(glob.glob(f'{ckpt_dir}/model_epoch*.pt')) if os.path.exists(ckpt_dir) else []

if existing:
    latest = existing[-1]
    m = re.search(r'model_epoch(\d+)\.pt', os.path.basename(latest))
    RESUME_EPOCH = int(m.group(1)) if m else 0
    print(f'✅ Found {len(existing)} checkpoint(s), latest at epoch {RESUME_EPOCH}')
else:
    RESUME_EPOCH = 0
    print('🆕 No previous checkpoints — will start fresh training')

## 4. Verify Setup

In [ ]:
import torch
import numpy as np

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

required = ['50Real.npy', '50Seq.npy', 'SeqCond.npy', 'RealCond.npy']
for f in required:
    path = f'data/{f}'
    if os.path.exists(path):
        print(f'  ✓ {f} ({os.path.getsize(path)/1024/1024:.1f} MB)')
    else:
        print(f'  ✗ {f} — MISSING!')

real = np.load('data/50Real.npy'); seq = np.load('data/50Seq.npy')
print(f'\n50Real.npy: {real.shape}'); print(f'50Seq.npy: {seq.shape}')

## 5. Train

Choose whether to resume from your latest checkpoint or start fresh.

In [ ]:
# Choose: resume or start fresh
resume = RESUME_EPOCH > 0  # auto-resume if checkpoints exist; set to False to force fresh

if resume:
    target_epochs = RESUME_EPOCH + 500  # always add 500 more
    print(f'▶️ Resuming from epoch {RESUME_EPOCH} → target {target_epochs}')
    !python src/train.py \
        --data_path='data' \
        --output='{DRIVE_OUTPUT}' \
        --max_epochs={target_epochs} \
        --auto \
        --T=1000 --ddim_steps=50 \
        --checkpoint_step=100 --vis_freq=10 \
        --resume
else:
    print(f'🆕 Starting fresh training → {DRIVE_OUTPUT}')
    !python src/train.py \
        --data_path='data' \
        --output='{DRIVE_OUTPUT}' \
        --max_epochs=500 \
        --auto \
        --T=1000 --ddim_steps=50 \
        --checkpoint_step=100 --vis_freq=10 \
        --yes

## 6. Monitor Training

Checkpoints (`.pt` files) saved to `{DRIVE_OUTPUT}/Checkpoints/`.
Sample animations (`.mp4`) saved to `{DRIVE_OUTPUT}/Samples/`.

In [ ]:
# List saved checkpoints
!ls -lh {DRIVE_OUTPUT}/Checkpoints/ 2>/dev/null || echo 'No checkpoints yet'

# List sample animations
!ls -lh {DRIVE_OUTPUT}/Samples/ 2>/dev/null || echo 'No samples yet'

## 7. View Generated Samples

In [ ]:
from IPython.display import Video
import glob

# Play the latest generated animation
samples = sorted(glob.glob(f'{DRIVE_OUTPUT}/Samples/*.mp4'))
if samples:
    print(f'Latest sample: {samples[-1]}')
    Video(samples[-1], width=500)
else:
    print('No samples found. Training may still be in progress.')

---

## 8. Launch Web UI (Optional)

After training, launch the Gradio interface to interactively sketch and generate plays:

In [ ]:
# Launch Gradio web UI — accessible via public link
!pip install -q gradio

# Find latest checkpoint
import glob
ckpts = sorted(glob.glob(f'{DRIVE_OUTPUT}/Checkpoints/model_epoch*.pt'))
checkpoint = ckpts[-1] if ckpts else None

if checkpoint:
    !python ui/app.py --checkpoint {checkpoint} --share &
    print(f'Using checkpoint: {checkpoint}')
else:
    print('No checkpoint — using random weights')
    !python ui/app.py --share &